# Evaluate Random Episode

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from contract import ROOT,REFERENCE,read_json,worker_command
from experiment_supervision import execute_stage
MODE='random'
def run_worker(resolved_config,stage_directory,simulator_seed,checkpoint=None,training_directory=None,action_seed=None):
    config=read_json(resolved_config)
    args=['--resolved',resolved_config,'--stage-dir',stage_directory,'--simulator-seed',simulator_seed]
    if MODE=='random':
        if action_seed is None:raise ValueError('Explicit action seed required')
        args+=['--action-seed',action_seed]
    else:
        args+=['--mode',MODE]
        if MODE=='evaluate':
            if checkpoint is None or training_directory is None:raise ValueError('Checkpoint and training identity required')
            args+=['--checkpoint',checkpoint,'--training-dir',training_directory]
    return execute_stage(worker_command('random_evaluation',args),Path(stage_directory),config['training_timeout_seconds'] if MODE=='train' else config['evaluation_timeout_seconds'],config['worker_id'])
print('Evaluate Random Episode definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Exclusive attempts and supervised worker processes definitions/execution completed.
Evaluate Random Episode definitions/execution completed.


In [3]:
result=read_json(REFERENCE/'runs/P1-RANDOM/attempt-01/evaluate-3001/result.json')
assert result['status']=='PASS'
print('Historical worker result (no new run):',json.dumps({k:result[k] for k in ['status','mode','pid','wall_seconds'] if k in result},indent=2))

Historical worker result (no new run): {
  "status": "PASS",
  "mode": "evaluate",
  "pid": 1119885,
  "wall_seconds": 6.994834381002875
}
